# **Project Name**    -  LLM Bot for E-commerce Customer Support using Fine Tuned Pre-Trained Model and RAG


##### **Project Type**    - Large Language Model
##### **Contribution**    - Individual


# **Project Summary -**

The project E-commerce Customer Support Conversational AI will involve creating an intelligent chatbot that will assist in improving the quality of customer support services provided in the e-commerce sector. Customers tend to ask several questions in regards to order tracking, refunds, returns, cancellations, payment and delivery. Answering these questions by human operators might take more time than needed, as well as creating extra workload for the customer support service providers.

The solution includes applying both a Large Language Model (LLM) and Retrieval-Augmented Generation (RAG) to give customers the most relevant and context-aware answers to their questions. This particular LLM is fine-tuned on the basis of an e-commerce customer support dataset in order to understand customer questions and generate conversational responses to them. RAG helps to retrieve the most relevant information from a knowledge base so that the chatbot could provide responses that have something to do with the documentation. An interface based on Streamlit will be used to make the system available via chat application.

# **GitHub Link -**

https://github.com/Someshwar24/ecommerce-customer-support-chatbot

# **Problem Statement**


An e-commerce business receives many customer inquiries regarding order tracking, refunds, returns, cancellations, payment methods, and shipping issues. The task of responding to these queries in time using manual customer services can be cumbersome, laborious, and hard to scale. Traditional chatbots might also have difficulty interpreting customer intent and providing contextually relevant responses. In such a scenario, the need arises for a smart customer support system which can comprehend natural language inputs and get relevant data from knowledge bases. The objective of the project is to design an AI-enabled conversational chatbot based on a fine-tuned Large Language Model and Retrieval-Augmented Generation (RAG).

# ***Let's Begin !***

In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

PyTorch: 2.15.0+cu132
CUDA Available: True


In [2]:
!pip install -q \
    langchain-core \
    langchain-community \
    langchain-text-splitters \
    sentence-transformers \
    faiss-cpu \
    transformers \
    datasets \
    accelerate \
    peft \
    trl \
    bitsandbytes \
    pypdf

In [3]:
from google.colab import files

uploaded = files.upload()

uploaded

Saving E_commerce_CustomerSupport.pdf to E_commerce_CustomerSupport (2).pdf


{'E_commerce_CustomerSupport (2).pdf': b'%PDF-1.4\n%\xe2\xe3\xcf\xd3\n10 0 obj\n<<\n/BaseFont /Helvetica\n/Encoding /WinAnsiEncoding\n/Name /F1\n/Subtype /Type1\n/Type /Font\n>>\nendobj\n11 0 obj\n<<\n/BaseFont /Helvetica-Bold\n/Encoding /WinAnsiEncoding\n/Name /F2\n/Subtype /Type1\n/Type /Font\n>>\nendobj\n12 0 obj\n<<\n/Filter [/ASCII85Decode /FlateDecode]\n/Length 1041\n>>\nstream\nGatm8bAQ&o\']&X:mVY-U_MXMTa"l*[MCC5r,+tFYcm32/TgY01@K`\'Zmj4eYG*$6b_:XKYM>)#X3TlhtJo;R2U\'(1cJ[Z^MckMoaBMNS]^;,M9o5d:!4JQ$cUad$Y<C#j=Q,<3l9b06/)9]>pZAPGH=f0OHbA1"Ro]2EkS\'BBS,V4&9QguE>8;>*)bXtbU/8%O[J_Pq_\\q"%tU]jf%Z#nI2p;<`2?)O2hn8@fl`Lh**%>/dJqubiYICC9PDPqh-iQp@%NKSVl2g3\'%hkBiHc\'s7EU`I?2P\\&Bj]<<4\'kXicQ,\\KiF$c4Z/,\\8rdk["OfaVo]j-?VZ@!XN+q\'$8D8/U]C(!\\?T]q_<d]$1t&Fn5hDD-<X[5Gp?\\XirSi7GtK(9=d2@CrEgsM]TPIQ>dZO+ke.^0VtuO:g@d&Lo?::Ok1\'dj,T]1(QI!okE#oo[$cobX+DgMKd?9QH8BFFk"Pgf[l4b>2!j-T!*hKXjp!kE$#<qSBG\\3!!Tk;F4p0^mK8pm1`([`7nWZlhW.5"5[Y(t1UHrs8X<YRFj!/E8c`4BTfVfCt4\\D1Gc51R<"%ogC=F"b`_Ob;$k>51\'J?u$&

In [4]:
from langchain_community.document_loaders import PyPDFLoader

pdf_file = list(uploaded.keys())[0]

loader = PyPDFLoader(pdf_file)

documents = loader.load()

print("Document loaded successfully!")

/tmp/ipykernel_40248/3487212871.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Document loaded successfully!


In [5]:
# Creating the chunks from uploaded file

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print("Chunks created successfully =",len(chunks))

Chunks created successfully = 604


In [6]:
# Embeddings

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Embedding model loaded successfully")

chunk_texts = [chunk.page_content for chunk in chunks]

print("Total text chunks:", len(chunk_texts))

print("\nFirst chunk:\n")
print(chunk_texts[0])

embeddings = embedding_model.encode(chunk_texts,show_progress_bar=True,convert_to_numpy=True)

print(f'Embedding shape: {embeddings.shape}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully
Total text chunks: 604

First chunk:

E-commerce Customer Support | WooCommerce Combined Knowledge Source
Page 1
 E-commerce Customer Support
 Combined WooCommerce Documentation
A consolidated educational knowledge source for the Customer Support RAG chatbot
Included topics
Order statuses  Refund processing  Order fulfillment and shipping tracking  Return and refund policy guidelines  Payment
methods and payment support
Scope and limitations


Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Embedding shape: (604, 384)


In [7]:
# Storing the embeddings

import faiss
import numpy as np

embeddings = np.array(
    embeddings,
    dtype=np.float32
)

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("FAISS index created successfully!")
print("Total vectors stored:", index.ntotal)

FAISS index created successfully!
Total vectors stored: 604


In [8]:
def retrieve_context(query, k=3):

    # Convert the user question into an embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype(np.float32)

    # Search FAISS for the top-k relevant chunks
    distances, indices = index.search(query_embedding, k)

    # Collect the retrieved document content
    retrieved_chunks = []

    for idx in indices[0]:
        retrieved_chunks.append(chunks[idx].page_content)

    # Combine retrieved chunks into one context
    context = "\n\n".join(retrieved_chunks)

    return context

In [9]:
query = "How can I get a refund for my order?"

context = retrieve_context(query)

print("User Question:")
print(query)

print("\nRetrieved Context:")
print(context)

User Question:
How can I get a refund for my order?

Retrieved Context:
 Open WooCommerce > Orders and select the order.
 Choose Refund in the order summary.
 Enter the quantity to refund for the relevant line items. Where inventory is not managed, a refund amount may be entered
without changing the product quantity.
 Add refund notes if needed, then select the gateway refund action showing the amount.
 Review the order status and order notes. A partial refund does not necessarily change the overall order status to
Refunded.
2.2 Verifying a refund

The merchant must separately return the funds through the provider's dashboard or another appropriate transfer method.
This may be needed for payment methods that do not support automatic refunds.
 Open WooCommerce > Orders and select the order.
 Choose Refund, enter the quantity or amount, and add notes if needed.
 Select the manual refund option.
 Separately return the funds through the payment provider or another appropriate proc

In [10]:
#LLM

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("CUDA Available:", torch.cuda.is_available())
print("Using Device:", device)

llm = llm.to(device)

print("LLM loaded successfully!")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


CUDA Available: True
Using Device: cuda
LLM loaded successfully!


In [11]:
def rag_response(query):

    # Retrieve relevant documents
    context = retrieve_context(query, k=3)

    # Construct an improved prompt
    prompt = f"""
      You are an e-commerce customer support assistant.

      Answer the customer's question using only the context below.

      Follow these instructions:
      1. Start with a direct answer.
      2. Present the steps in a clear numbered sequence.
      3. Explain automatic and manual refunds separately when relevant.
      4. Avoid repeating the same information.
      5. Use simple customer-friendly language.
      6. Do not invent information that is not in the context.
      7. If information is unavailable, clearly mention it.

      Context:
      {context}

      Customer Question:
      {query}

      Answer:
      """


    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    outputs = llm.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        num_beams=4
    )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return response

In [12]:
query = "How can I get a refund for my order?"

answer = rag_response(query)

print("User Question:", query)
print("\nRAG Generated Answer:", answer)

/usr/local/lib/python3.13/dist-packages/triton/knobs.py:504: UserWarning: TRITON_DEFAULT_FP_FUSION is unset: defaulting to FP fusion ENABLED for backward compatibility in Triton 3.9. The default will change to DISABLED in Triton 3.10; set TRITON_DEFAULT_FP_FUSION=1 to keep fusion enabled or =0 to adopt the new default early and silence this warning.
  warn_once(FP_FUSION_DEFAULT_WARNING)


User Question: How can I get a refund for my order?

RAG Generated Answer: Automatic refunds Open WooCommerce > Orders and select the order. Choose Refund in the order summary. Enter the quantity to refund for the relevant line items. Add refund notes if needed, then select the gateway refund action showing the amount. Review the order status and order notes. A partial refund does not necessarily change the overall order status to Refunded. 2.2 Verifying a refund The merchant must separately return the funds through the provider's dashboard or another appropriate transfer method. This may be needed for payment methods that do not support automatic refunds. Open WooCommerce > Orders and select the order. Choose Refund, enter the quantity or amount, and add notes if needed. Select the manual refund option. Separately return the funds through the payment provider or another appropriate process.


In [13]:
test_queries = [
    "How can I get a refund for my order?",
    "Does cancelling an order automatically refund the money?",
    "What does Pending payment mean?",
    "What is the difference between automatic and manual refunds?",
    "How can a merchant add tracking information?"
]

rag_results = []

for query in test_queries:

    answer = rag_response(query)

    rag_results.append({
        "Question": query,
        "Answer": answer
    })

    print("\nCustomer Question:", query)
    print("Chatbot Answer:", answer)
    print("-" * 80)


Customer Question: How can I get a refund for my order?
Chatbot Answer: Automatic refunds Open WooCommerce > Orders and select the order. Choose Refund in the order summary. Enter the quantity to refund for the relevant line items. Add refund notes if needed, then select the gateway refund action showing the amount. Review the order status and order notes. A partial refund does not necessarily change the overall order status to Refunded. 2.2 Verifying a refund The merchant must separately return the funds through the provider's dashboard or another appropriate transfer method. This may be needed for payment methods that do not support automatic refunds. Open WooCommerce > Orders and select the order. Choose Refund, enter the quantity or amount, and add notes if needed. Select the manual refund option. Separately return the funds through the payment provider or another appropriate process.
--------------------------------------------------------------------------------

Customer Questi

In [14]:
import pandas as pd

rag_results_df = pd.DataFrame(rag_results)

rag_results_df.head()

,Question,Answer
0,How can I get a refund for my order?,Automatic refunds Open WooCommerce > Orders an...
1,Does cancelling an order automatically refund ...,Changing an order status directly to Cancelled...
2,What does Pending payment mean?,The order has been received but payment has no...
3,What is the difference between automatic and m...,Automatic refunds Open WooCommerce > Orders an...
4,How can a merchant add tracking information?,Create a new fulfillment or edit an existing o...


In [15]:
rag_results_df.to_csv(
    "pretrained_rag_baseline.csv",
    index=False
)

In [16]:
# FINE TUNING A PRE-TRAINED LLM MODEL

!pip install -q datasets

In [17]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset(
    "bitext/Bitext-customer-support-llm-chatbot-training-dataset"
)

df = dataset["train"].to_pandas()

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (26872, 5)


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


In [18]:
print("Columns:", df.columns.tolist())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDataset Information:")
df.info()

Columns: ['flags', 'instruction', 'category', 'intent', 'response']

Missing Values:
flags          0
instruction    0
category       0
intent         0
response       0
dtype: int64

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26872 entries, 0 to 26871
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   flags        26872 non-null  object
 1   instruction  26872 non-null  object
 2   category     26872 non-null  object
 3   intent       26872 non-null  object
 4   response     26872 non-null  object
dtypes: object(5)
memory usage: 1.0+ MB


In [19]:
print("Number of Categories:", df["category"].nunique())

print("\nCategories:")
print(df["category"].unique())

print("\nNumber of Intents:", df["intent"].nunique())

print("\nTop 10 Intents:")
print(df["intent"].value_counts().head(10))

print("\nDuplicate Rows:", df.duplicated().sum())

print("\nDuplicate Instructions:", df["instruction"].duplicated().sum())

Number of Categories: 11

Categories:
['ORDER' 'SHIPPING' 'CANCEL' 'INVOICE' 'PAYMENT' 'REFUND' 'FEEDBACK'
 'CONTACT' 'ACCOUNT' 'DELIVERY' 'SUBSCRIPTION']

Number of Intents: 27

Top 10 Intents:
intent
contact_customer_service    1000
complaint                   1000
check_invoice               1000
switch_account              1000
edit_account                1000
contact_human_agent          999
check_payment_methods        999
delivery_period              999
newsletter_subscription      999
get_invoice                  999
Name: count, dtype: int64

Duplicate Rows: 0

Duplicate Instructions: 2237


In [20]:
training_df = df[["instruction", "response"]].copy()

training_df.rename(
    columns={
        "instruction": "input",
        "response": "output"
    },
    inplace=True
)

training_df.head()

,input,output
0,question about cancelling order {{Order Number}},I've understood you have a question regarding ...
1,i have a question about cancelling oorder {{Or...,I've been informed that you have a question ab...
2,i need help cancelling puchase {{Order Number}},I can sense that you're seeking assistance wit...
3,I need to cancel purchase {{Order Number}},I understood that you need assistance with can...
4,"I cannot afford this order, cancel purchase {{...",I'm sensitive to the fact that you're facing f...


In [21]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    training_df,
    test_size=0.1,
    random_state=42
)

print("Training Samples:", len(train_df))
print("Validation Samples:", len(val_df))

Training Samples: 24184
Validation Samples: 2688


In [22]:
train_df, test_df = train_test_split(
    train_df,
    test_size=1/9,
    random_state=42
)

print("Training Samples:", len(train_df))
print("Validation Samples:", len(val_df))
print("Testing Samples:", len(test_df))

Training Samples: 21496
Validation Samples: 2688
Testing Samples: 2688


In [23]:
from transformers import AutoTokenizer

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded successfully!")

Tokenizer loaded successfully!


In [24]:
import torch
from transformers import AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

model = model.to(device)

print("Model loaded successfully!")
print("Device:", device)
print("CUDA available:", torch.cuda.is_available())

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model loaded successfully!
Device: cuda
CUDA available: True


In [25]:
import torch
from transformers import AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
base_model = base_model.to(device)

print("Model loaded:", model_name)
print("Device:", device)
print("Existing PEFT configuration:",
      hasattr(base_model, "peft_config"))

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Base model loaded: cuda:0


In [26]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q", "v"],
    bias="none"
)

model = get_peft_model(base_model, lora_config)

model.print_trainable_parameters()

trainable params: 884,736 || all params: 248,462,592 || trainable%: 0.3561


In [29]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    padding=True
)

print("Data collator created successfully!")

Data collator created successfully!


In [31]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_df,
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_df,
    preserve_index=False
)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

Training samples: 21496
Validation samples: 2688


In [32]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
val_dataset = Dataset.from_pandas(val_df, preserve_index=False)

def tokenize_function(examples):

    inputs = tokenizer(
        examples["input"],
        max_length=128,
        truncation=True
    )

    labels = tokenizer(
        text_target=examples["output"],
        max_length=256,
        truncation=True
    )

    inputs["labels"] = labels["input_ids"]

    return inputs


tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["input", "output"]
)

tokenized_val = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["input", "output"]
)

Map:   0%|          | 0/21496 [00:00<?, ? examples/s]

Map:   0%|          | 0/2688 [00:00<?, ? examples/s]

In [37]:
import torch
from transformers import AutoModelForSeq2SeqLM
from peft import get_peft_model, LoraConfig, TaskType

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-base"
).to("cuda")

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q", "v"],
    bias="none"
)

model = get_peft_model(base_model, lora_config)

print("Fresh LoRA model created!")
model.print_trainable_parameters()

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Fresh LoRA model created!
trainable params: 884,736 || all params: 248,462,592 || trainable%: 0.3561


In [39]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./flan_t5_lora_ecommerce",

    num_train_epochs=1,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    learning_rate=2e-4,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=100,

    fp16=False,
    bf16=False,

    predict_with_generate=False,
    report_to="none"
)

print("Epochs:", training_args.num_train_epochs)
print("Batch size:", training_args.per_device_train_batch_size)
print("Learning rate:", training_args.learning_rate)
print("FP16:", training_args.fp16)

Epochs: 1
Batch size: 16
Learning rate: 0.0002
FP16: False


In [40]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    processing_class=tokenizer
)

print("Trainer created successfully!")
print("Training samples:", len(trainer.train_dataset))
print("Validation samples:", len(trainer.eval_dataset))
print("FP16:", trainer.args.fp16)

Trainer created successfully!
Training samples: 21496
Validation samples: 2688
FP16: False


In [41]:
train_result = trainer.train()

Epoch,Training Loss,Validation Loss
1,1.640551,1.371021


In [42]:
save_path = "./flan_t5_lora_ecommerce"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("LoRA adapter saved successfully!")
print("Location:", save_path)

LoRA adapter saved successfully!
Location: ./flan_t5_lora_ecommerce


In [43]:
model.eval()

test_questions = [
    "How can I request a refund for my order?",
    "What payment methods are available?",
    "How can I track my order?"
]

for question in test_questions:

    inputs = tokenizer(
        question,
        return_tensors="pt",
        max_length=128,
        truncation=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            num_beams=4,
            early_stopping=True
        )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    print("Customer:", question)
    print("Chatbot:", response)
    print("-" * 70)

Customer: How can I request a refund for my order?
Chatbot: I'm here to assist you in requesting a refund for your order. To request a refund, please follow these steps: 1. Log in to your account on our website. 2. Navigate to the "My Account" or "My Account" section. 3. Look for the "Refund" or "Refund" option. 4. Click on the "Refund" or "Refund" button. 5. Click on the "Refund" or "Refund" button. 6.
----------------------------------------------------------------------
Customer: What payment methods are available?
Chatbot: We're here to assist you with payment methods. We offer a variety of payment methods, including Visa, MasterCard, Discover, and American Express. We offer a variety of payment methods, including credit cards, debit cards, and a variety of other payment methods. Please let us know if you have any questions or need further assistance.
----------------------------------------------------------------------
Customer: How can I track my order?
Chatbot: Thank you for re

In [54]:
query = "How can I request a refund for my order?"

# Retrieve relevant documents
query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True
).astype("float32")

distances, indices = index.search(query_embedding, k=3)

# Combine retrieved text
context = "\n".join(
    chunks[idx].page_content
    for idx in indices[0]
)

# Create prompt
prompt = f"""
Extract the refund instructions from the context.
Do not add any information.
Do not write a general customer support response.

Knowledge base:
{context}

Customer question:
{query}

Answer:
"""

# Generate response
inputs = tokenizer(
    prompt,
    return_tensors="pt",
    max_length=512,
    truncation=True
).to(model.device)

model.eval()

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        num_beams=4,
        early_stopping=True
    )

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Customer:", query)
print("\nRAG Context:", context[:500])
print("\nChatbot Response:", response)

Customer: How can I request a refund for my order?

RAG Context:  Open WooCommerce > Orders and select the order.
 Choose Refund in the order summary.
 Enter the quantity to refund for the relevant line items. Where inventory is not managed, a refund amount may be entered
without changing the product quantity.
 Add refund notes if needed, then select the gateway refund action showing the amount.
 Review the order status and order notes. A partial refund does not necessarily change the overall order status to
Refunded.
2.2 Verifying a refund
E-commerce Cu

Chatbot Response: Thank you for reaching out! I'm here to guide you through the process of requesting a refund. If you encounter any difficulties or have any further questions, please don't hesitate to reach out to our customer support team. We're here to assist you every step of the way.


In [53]:
query = "How can I request a refund for my order?"

context = chunks[indices[0][0]].page_content

prompt = f"""
Extract the refund instructions from the context.
Do not add any information.
Do not write a general customer support response.

{context}

Refund instructions:
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        num_beams=2,
        do_sample=False
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

We're here to assist you in determining the refund instructions from the context of your order. To ensure that you're able to review the refund instructions, you can visit our website and navigate to the "Order Summary" section. This section will provide you with detailed instructions on how to proceed. If you encounter any difficulties or have any further questions, please don't hesitate to reach out to our customer support team. We're here to assist you every step of the way.


# **Conclusion**

The E-commerce Customer Support Conversational AI project considers the prospects of using Large Language Models in combination with Retrieval-Augmented Generation to solve problems related to customer service. The integration of domain-specific fine-tuning and knowledge-based information retrieval helps to create more precise and accessible responses to customer inquiries. In addition, the presented project is an example of applying generative AI technologies to real-life situations via the chatbot interface. Even though the system needs to be properly evaluated in order to guarantee the precision and reliability of responses, it provides great possibilities for simplifying the process of customer service and making it more effective.